In [0]:
data = [
    (1, 101, 500.0, "2024-01-01"),
    (2, 102, 200.0, "2024-01-02"),
    (3, 101, 300.0, "2024-01-03"),
    (4, 103, 100.0, "2024-01-04"),
    (5, 102, 400.0, "2024-01-05"),
    (6, 103, 600.0, "2024-01-06"),
    (7, 101, 200.0, "2024-01-07"),
]

columns = ["transaction_id", "user_id", "transaction_amount", "transaction_date"]

df = spark.createDataFrame(data, columns)
display(df)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

window_spec = Window.partitionBy('user_id').orderBy(desc('transaction_date'))

df_top3 = df.groupBy('user_id').agg(sum('transaction_amount').alias("total_amount")).orderBy(desc("total_amount")).limit(3)
#df_top3.show()

df_tdata = df.withColumn('row_num',row_number().over(window_spec)).filter(col('row_num')==1).select("user_id","transaction_date")
#df_tdata.show()

df_final = df_top3.join(df_tdata,on="user_id")
df_final.show()

In [0]:
#simple method, stupid fellow

df_final = df.groupBy("user_id").agg(
    sum("transaction_amount").alias('total_amount'),
    max('transaction_date').alias('max_transaction_date')
).orderBy(desc(col("total_amount"))).limit(3)
df_final.show()